In [10]:
from pathlib import Path
import numpy as np
import h5py
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from astroclip.models.astroclip import AstroClipModel
from astroclip.data.datamodule import AstroClipCollator
from datasets import load_from_disk

# ---------------------------------------------------------------------
ASTROCLIP_ROOT = Path("/Users/marchuertascompany/Documents/teaching/2025_astroinfo/data")
CKPT = ASTROCLIP_ROOT / "astroclip.ckpt"
DATASET = ASTROCLIP_ROOT 
OUT_DIR = ASTROCLIP_ROOT 
OUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

ds = load_from_disk(DATASET)                       # expects a save_to_disk dataset dict
collator = AstroClipCollator(center_crop=144)
model = AstroClipModel.load_from_checkpoint(str(CKPT)).eval().to(device)

class ArrayDataset(Dataset):
    def __init__(self, hf_dataset):
        self.dataset = hf_dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        record = dict(self.dataset[idx])
        record["image"] = np.asarray(record["image"], dtype=np.float32)
        record["spectrum"] = np.asarray(record["spectrum"], dtype=np.float32)
        return record

def embed_split(split_name: str, modality: str, output_path: Path):
    loader = DataLoader(
        ArrayDataset(ds[split_name]),
        batch_size=128,
        shuffle=False,
        collate_fn=collator,
    )

    embeddings, redshifts, targetids = [], [], []

    for batch in tqdm(loader, desc=f"{modality} {split_name}"):
        imgs = batch["image"]
        if not isinstance(imgs, torch.Tensor):
            imgs = torch.tensor(imgs, dtype=torch.float32)
        if imgs.ndim == 4 and imgs.shape[-1] == 3:
            imgs = imgs.permute(0, 3, 1, 2)         # convert HWC → CHW
        imgs = imgs.to(device)

        specs = batch["spectrum"]
        if not isinstance(specs, torch.Tensor):
            specs = torch.tensor(specs, dtype=torch.float32)
        specs = specs.to(device)

        with torch.no_grad():
            if modality == "image":
                emb = model(imgs, input_type="image")
            elif modality == "spectrum":
                emb = model(specs, input_type="spectrum")
            else:
                raise ValueError("modality must be 'image' or 'spectrum'")

        embeddings.append(emb.cpu().numpy())
        redshifts.append(np.asarray(batch["redshift"], dtype=np.float32))
        if "targetid" in batch:
            targetids.append(np.asarray(batch["targetid"], dtype=np.int64))

    embeddings = np.concatenate(embeddings, axis=0)
    redshifts  = np.concatenate(redshifts, axis=0)
    targetids  = np.concatenate(targetids, axis=0) if targetids else None

    with h5py.File(output_path, "w") as f:
        f.create_dataset(f"{modality}_embeddings", data=embeddings)
        f.create_dataset("redshift", data=redshifts)
        if targetids is not None:
            f.create_dataset("targetid", data=targetids)

    print(f"Saved {modality} {split_name} embeddings to {output_path}")

embed_split("train", "image",    OUT_DIR / "embeddings_image_train.h5")
embed_split("test",  "image",    OUT_DIR / "embeddings_image_test.h5")
embed_split("train", "spectrum", OUT_DIR / "embeddings_spectrum_train.h5")
embed_split("test",  "spectrum", OUT_DIR / "embeddings_spectrum_test.h5")


Using device: mps


image train: 100%|██████████| 8/8 [02:28<00:00, 18.55s/it]


Saved image train embeddings to /Users/marchuertascompany/Documents/teaching/2025_astroinfo/data/embeddings_image_train.h5


image test: 100%|██████████| 2/2 [00:32<00:00, 16.25s/it]


Saved image test embeddings to /Users/marchuertascompany/Documents/teaching/2025_astroinfo/data/embeddings_image_test.h5


spectrum train: 100%|██████████| 8/8 [02:45<00:00, 20.74s/it]


Saved spectrum train embeddings to /Users/marchuertascompany/Documents/teaching/2025_astroinfo/data/embeddings_spectrum_train.h5


spectrum test: 100%|██████████| 2/2 [00:36<00:00, 18.00s/it]


Saved spectrum test embeddings to /Users/marchuertascompany/Documents/teaching/2025_astroinfo/data/embeddings_spectrum_test.h5


In [8]:
from tqdm.auto import tqdm
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

model = AstroClipModel.load_from_checkpoint(str(CKPT)).eval().to(device)

loader = DataLoader(
    ArrayDataset(ds["test"]),   # wrapper that converts HF records to NumPy/tensors
    batch_size=128,
    shuffle=False,
    collate_fn=collator,
)

it_loader = iter(loader)
max_batches = 20
embeddings, redshift, images = [], [], []

for batch_idx, batch in enumerate(loader):
    if batch_idx >= max_batches:
        break
    batch = next(it_loader)
    imgs = batch["image"]
    if not isinstance(imgs, torch.Tensor):
        imgs = torch.tensor(np.asarray(imgs), dtype=torch.float32)
    if imgs.ndim == 4 and imgs.shape[-1] == 3:        # channel-last? convert once
        imgs = imgs.permute(0, 3, 1, 2)
    imgs = imgs.to(device)

    with torch.no_grad():
        emb = model(imgs, input_type="image").cpu()

    embeddings.append(emb.numpy())
    redshift.append(batch["redshift"])
    images.append(batch["image"])                     # keep original structure

embeddings = np.concatenate(embeddings)
redshift = np.concatenate(redshift)
images = np.concatenate(images)


Using device: mps
